# Wanda 2:4 Sparsity + vLLM — Llama-3.1-8B *and* Mistral-7B production-style baseline

Companion to `colab_standalone.ipynb`. Same Drive layout, same accuracy task (PIQA), same fixed latency prompts — different sparsity method (Wanda 2:4 weight pruning) and different inference engine (vLLM with native Sparse Tensor Core kernels).

**Switch model families with one line in §6.** Run the notebook end-to-end once with `MODEL_FAMILY = 'llama3'`, then once more with `MODEL_FAMILY = 'mistral'`. Result JSONs persist on Drive and §13 reads whatever's there.

**What this notebook produces** (per family):
- `<family>_dense_vllm.json` — dense base model served via vLLM
- `<family>_wanda_2of4_vllm.json` — Wanda-pruned 2:4 sparse model served via vLLM
- A six-way comparison printed at the end (HF dense, R-Sparse, vLLM dense, Wanda vLLM, per family).

**Why Wanda + 2:4 + vLLM:** R-Sparse's 43% speedup needs custom kernels not in the open-source release (see `docs/r_sparse_latency_analysis.md`). 2:4 structured weight sparsity has native NVIDIA Sparse Tensor Core support, and **vLLM dispatches to it automatically** when the checkpoint declares 2:4 sparsity in its `compressed-tensors` metadata.\n\n**Note on Path B vs kernel speedup:** Path A (pre-sparsified) gets the ~1.5× kernel speedup because RedHatAI's checkpoint declares 2:4 sparsity in its config. Path B (run-Wanda-yourself) saves dense fp16 with the 2:4 zero pattern — vLLM can\'t auto-detect the pattern from dense shards, so tok/s ≈ dense. Path B measures the Wanda *algorithm* (accuracy), Path A measures the *kernel* (speed). §14 expands.

## Two paths in §6 — and they live in DIFFERENT Colab runtimes

| | **Path A — pre-sparsified (default)** | **Path B — run Wanda yourself** |
|---|---|---|
| time | ~5 min | ~45-90 min on A100 |
| MODE in §6 | `'PRE_SPARSIFIED'` | `'RUN_WANDA'` |
| install | §4 only (`vllm`, `lm-eval[vllm]`, `ray`) | §8 inline (`llmcompressor`) |
| where | one runtime end-to-end | TWO runtimes: sparsifier → eval |

`vllm` and `llmcompressor` pin incompatible `torch` (2.11 vs ≤2.10) and `compressed-tensors` (0.15 vs 0.14) versions. They CANNOT live in the same runtime. **Path B requires two runtimes:**

1. **Sparsifier runtime** — fresh runtime; §1, §2, §3, §5, §6 (with `MODE='RUN_WANDA'`), §8. **Skip §4.** Wanda saves the sparse checkpoint to Drive. Then **Runtime → Disconnect and delete runtime**.
2. **Eval runtime** — fresh runtime; §1-§7 (still `MODE='RUN_WANDA'`), §10-§13. **Skip §8.** §7 picks up the checkpoint your sparsifier runtime saved.

If you stick with the default (Path A), ignore all of the above — just run the cells top-to-bottom.

---

**What changed from the original notebook:**
- Dense baseline is now `meta-llama/Llama-3.1-8B` (BASE, 3.1) so it matches the family of `RedHatAI/Sparse-Llama-3.1-8B-2of4`. Original used `Meta-Llama-3-8B-Instruct` (3.0 + Instruct), which mixed three changes (3.0→3.1, instruct→base, dense→sparse) into one comparison.
- §4 install dropped `--force-reinstall --no-deps` and the `llmcompressor` line. Pre-pinning version-sensitive packages and then letting vllm/lm-eval re-resolve was leaving torch in a half-broken state on disk.
- `LIMIT` defaults to `None` (full task). `--limit 64` had stderr ≈ 0.05 — too wide to detect a real sparsity effect.

## §1 Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## §2 GPU sanity-check

vLLM 2:4 sparse requires Ampere or newer (Sparse Tensor Cores). T4 is **not** Ampere — you'll need at least an A100 / A10 / L4 / H100. Pro+ usually gives A100; verify here.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU'
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
print(f'GPU:      {name}')
print(f'Compute:  sm_{cap[0]}{cap[1]}  (need >= sm_80 for 2:4 Sparse Tensor Cores)')
print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
if cap[0] < 8:
    print('\n⚠️  Pre-Ampere GPU detected. vLLM will run dense, no 2:4 acceleration.')
    print('   Switch runtime to A100 (Runtime > Change runtime type > A100 GPU).')

## §3 Persistent paths (matches `colab_standalone.ipynb`)

Result JSONs go to the same `results/` dir as the R-Sparse runs so the existing comparison docs can read them.

In [ ]:
import os
from pathlib import Path

PERSIST_ROOT = Path('/content/drive/MyDrive/r_sparse')
HF_CACHE     = PERSIST_ROOT / 'hf_cache'
RESULTS_DIR  = PERSIST_ROOT / 'results'
WANDA_OUT    = PERSIST_ROOT / 'wanda_checkpoints'   # only used by Path B
for d in (PERSIST_ROOT, HF_CACHE, RESULTS_DIR, WANDA_OUT):
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_CACHE)
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'transformers')

for k in ('HF_HOME', 'HF_DATASETS_CACHE', 'TRANSFORMERS_CACHE'):
    print(f'{k:22s} = {os.environ[k]}')
print(f'\nRESULTS_DIR  = {RESULTS_DIR}')
print(f'WANDA_OUT    = {WANDA_OUT}  (only used if you run Path B)')

## §4 Install dependencies (eval runtime)

vLLM 0.20 ships sparse-checkpoint loading via `compressed-tensors`. `lm-eval[vllm]` is the accuracy harness with vLLM backend. `ray` is required because lm-eval's vLLM backend imports `ray` at module load (even for single-GPU runs).

**Don't install `llmcompressor` here.** It pins `torch<=2.10` / `compressed-tensors==0.14`, which conflicts with vllm's `torch==2.11` / `compressed-tensors==0.15`. Path B installs `llmcompressor` in §8 inside a separate "sparsifier" runtime — never alongside vllm.

If the import below fails with `ImportError: cannot import name 'InlinedCodeCache' from torch._guards`, you have a half-broken torch on disk from a previous force-reinstall. **Runtime → Disconnect and delete runtime** (a plain "Restart session" is not enough — it keeps the broken site-packages), open a fresh runtime, and run only this cell.

In [ ]:
%pip install -q --upgrade pip

# Wipe torch fully before installing vllm. Colab's preinstalled torch
# and the vllm-pulled torch==2.11.0 wheel from PyPI don't always have
# the same internal file layout — pip overlays the new files but leaves
# orphans from the old ones, producing ImportError on torch internals
# like InlinedCodeCache or is_opaque_value. pip uninstall + rm -rf the
# directory is the only reliable way to start clean.
%pip uninstall -y -q torch torchvision torchaudio
import shutil, glob
for pat in ['/usr/local/lib/python3.12/dist-packages/torch',
            '/usr/local/lib/python3.12/dist-packages/torch-*.dist-info',
            '/usr/local/lib/python3.12/dist-packages/torchvision*',
            '/usr/local/lib/python3.12/dist-packages/torchaudio*']:
    for d in glob.glob(pat):
        shutil.rmtree(d, ignore_errors=True)

# Now install vllm. It pulls a self-consistent torch / transformers /
# datasets / huggingface_hub / tokenizers / compressed-tensors. ray is
# required because lm-eval's vLLM backend imports it at module load.
# llmcompressor is NOT installed here — it pins torch<=2.10 /
# compressed-tensors==0.14, conflicting with vllm. Path B installs
# llmcompressor in §8 inside a separate "sparsifier" runtime.
%pip install -q --no-cache-dir "vllm==0.20.1" "lm-eval[vllm]>=0.4.4" "ray>=2.9"

# If torch was already imported in this kernel, sys.modules still holds
# the old broken instance even though disk is now clean. In that case,
# restart the kernel and re-run this cell — the second run's pip is a
# no-op and the import block below succeeds.
import sys
if 'torch' in sys.modules:
    print('-' * 60)
    print('Torch was already loaded in this kernel. Restart now:')
    print('  Runtime → Restart session  (Ctrl/Cmd-M, .)')
    print('then re-run this cell. (Pip will be a no-op the second time.)')
    print('-' * 60)
else:
    import torch, transformers, datasets, vllm
    print(f'torch:        {torch.__version__}')
    print(f'transformers: {transformers.__version__}')
    print(f'datasets:     {datasets.__version__}')
    print(f'vllm:         {vllm.__version__}')
    print()
    print('Ignore cudf-cu12 / cuml-cu12 / pylibraft / cuda-python /')
    print("cuda-toolkit / opentelemetry / google-adk / ipython conflict")
    print("warnings in the pip output — that's Colab's preinstalled")
    print('package set, unused by this notebook.')

## §5 HuggingFace login

Llama-3-8B-Instruct is gated. Pre-sparsified Neural Magic / Red Hat AI checkpoints are usually open. Either way, log in once.

In [ ]:
from huggingface_hub import login
login()

## §6 Choose path: pre-sparsified vs run-Wanda-yourself

**Path A (default, recommended for first run):** download a pre-pruned 2:4 sparse Llama-3.1 / Mistral checkpoint from HuggingFace Hub. ~5 min, zero engineering.

**Path B:** run Wanda yourself via `llmcompressor` in a **separate "sparsifier" runtime**, save the result to Drive, then come back to a fresh "eval" runtime to score it. ~45-90 min including calibration.

Edit `MODEL_FAMILY` and `MODE` below.

In [ ]:
MODEL_FAMILY = 'llama3'           # 'llama3' or 'mistral'
MODE         = 'PRE_SPARSIFIED'   # 'PRE_SPARSIFIED' (recommended) or 'RUN_WANDA'

TASK   = 'piqa'
LIMIT  = None    # None = full task (~1.8k examples for piqa, a few minutes
                 # on A100). Set to 64 only as a smoke test — stderr ~0.05
                 # at that limit, which is too wide to detect a real
                 # sparsity effect.

# Per-family config. The dense baseline must match the sparse model's exact
# family (same major version, same base-vs-instruct) — otherwise the
# "sparsity effect" measurement is contaminated by the family difference.
#
# Original notebook paired Meta-Llama-3-8B-Instruct (3.0, instruct) with
# RedHatAI/Sparse-Llama-3.1-8B-2of4 (3.1, base). That mixed three changes
# into one comparison: 3.0->3.1, instruct->base, dense->2:4. Now both
# sides are 3.1 base, so only sparsity differs.
FAMILY_CONFIG = {
    'llama3': {
        'dense_hf_id': 'meta-llama/Llama-3.1-8B',
        'pre_sparsified_candidates': [
            'RedHatAI/Sparse-Llama-3.1-8B-2of4',
            'neuralmagic/Sparse-Llama-3.1-8B-2of4',
        ],
        'wanda_subdir': 'llama31-8b-wanda-2of4',
    },
    'mistral': {
        'dense_hf_id': 'mistralai/Mistral-7B-v0.3',
        'pre_sparsified_candidates': [
            'RedHatAI/Sparse-Mistral-7B-2of4',
            'neuralmagic/Sparse-Mistral-7B-Instruct-v0.3-2of4',
            'nm-testing/Mistral-7B-Instruct-v0.3-pruned-50-2of4',
        ],
        'wanda_subdir': 'mistral-7b-v0.3-wanda-2of4',
    },
}

cfg = FAMILY_CONFIG[MODEL_FAMILY]
DENSE_HF_ID                = cfg['dense_hf_id']
PRE_SPARSIFIED_CANDIDATES  = cfg['pre_sparsified_candidates']
WANDA_OUTPUT_DIR           = WANDA_OUT / cfg['wanda_subdir']

print(f'Family:             {MODEL_FAMILY}')
print(f'Mode:               {MODE}')
print(f'Dense baseline:     {DENSE_HF_ID}')
print(f'Task / limit:       {TASK} / {LIMIT if LIMIT is not None else "full"}')
if MODE == 'PRE_SPARSIFIED':
    print(f'Sparse candidates:')
    for c in PRE_SPARSIFIED_CANDIDATES:
        print(f'    {c}')
elif MODE == 'RUN_WANDA':
    print(f'Wanda output dir:   {WANDA_OUTPUT_DIR}')
else:
    raise ValueError(f'Unknown MODE: {MODE}')

## §7 Resolve the sparse model id

Path A: probe the candidate list, take the first one HF accepts. Path B: run Wanda; output path becomes the sparse model id.

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.errors import RepositoryNotFoundError, GatedRepoError

def first_available(candidates):
    api = HfApi()
    for cid in candidates:
        try:
            api.model_info(cid)
            return cid
        except (RepositoryNotFoundError, GatedRepoError) as exc:
            print(f'  {cid:45s}  unavailable ({type(exc).__name__})')
        except Exception as exc:
            print(f'  {cid:45s}  error: {exc}')
    return None

if MODE == 'PRE_SPARSIFIED':
    print('Probing pre-sparsified checkpoints on Hub:')
    SPARSE_HF_ID = first_available(PRE_SPARSIFIED_CANDIDATES)
    if SPARSE_HF_ID is None:
        raise RuntimeError(
            'None of the candidate pre-sparsified checkpoints are reachable. '
            'Either fix the candidate list (search HuggingFace for current 2of4 '
            'Llama checkpoints — Neural Magic / RedHatAI are the active publishers) '
            'or switch MODE to RUN_WANDA in §6.'
        )
    print(f'\n✓ Using: {SPARSE_HF_ID}')
elif MODE == 'RUN_WANDA':
    SPARSE_HF_ID = str(WANDA_OUTPUT_DIR)
    if not (WANDA_OUTPUT_DIR / 'config.json').exists():
        print(f'No Wanda output yet at {WANDA_OUTPUT_DIR}.')
        print('Run §8 (Wanda pruning) before continuing.')
    else:
        print(f'Reusing existing Wanda checkpoint at {WANDA_OUTPUT_DIR}')

## §8 [Path B only, sparsifier runtime] Run Wanda via `llmcompressor`

**Skip this entirely if `MODE='PRE_SPARSIFIED'`.** The pre-sparsified checkpoint already lives on the Hub.

If `MODE='RUN_WANDA'`: run this cell ONLY in a fresh "sparsifier" runtime where `vllm` has not been installed. Cell guards against running it in the eval runtime — if it detects torch 2.11 (vllm's torch) or vllm in `sys.modules`, it raises and refuses to install `llmcompressor`. The two pin incompatible versions of torch and `compressed-tensors`; co-installing them silently corrupts both.

Wanda's metric is one line: prune the smallest `|W_ij| · ‖X_j‖₂` per output row. The 2:4 mask structure constrains top-k selection to keep exactly 2 of every 4 contiguous weights — the layout NVIDIA Sparse Tensor Cores require. `llmcompressor`'s `oneshot()` API wraps it in a single call.

Calibration: 512 samples from `open_platypus`, sequence length 2048. ~30-60 min on A100.

**After this cell finishes** the checkpoint is on Drive at `WANDA_OUTPUT_DIR`. Disconnect and delete this runtime, then open a fresh one and run §1-§4 (pulls vllm) + §5-§7 (re-resolves `SPARSE_HF_ID` to the dir you just saved) + §10-§13 (eval).

In [ ]:
if MODE != 'RUN_WANDA':
    print(f'Path A (pre-sparsified) — skipping Wanda. Sparse model id: {SPARSE_HF_ID}')
else:
    # =====================================================================
    # WARNING — RUN THIS IN A DEDICATED "SPARSIFIER" RUNTIME ONLY.
    # =====================================================================
    # llmcompressor pins torch<=2.10 / compressed-tensors==0.14.
    # vllm pins torch==2.11 / compressed-tensors==0.15.
    # They CANNOT coexist. If you run this cell in the same runtime as
    # §11/§12 eval, vllm and llmcompressor both end up broken with
    # ImportError on the next import.
    #
    # Correct workflow:
    #   1. Sparsifier runtime: §1, §2, §3, §5, §6, §8 (this cell). SKIP §4.
    #      Saves the sparse checkpoint to Drive.
    #   2. Disconnect AND DELETE the runtime.
    #   3. Eval runtime: fresh runtime; §1-§7, §10-§13. §7 picks up the
    #      checkpoint you saved in step 1 (because MODE is still RUN_WANDA).
    # =====================================================================
    import sys, torch
    if 'vllm' in sys.modules or torch.__version__.startswith('2.11'):
        raise RuntimeError(
            'Refusing to install llmcompressor: this runtime already has '
            'vllm / torch 2.11 loaded. Installing llmcompressor on top of '
            'that will break both. Runtime → Disconnect and delete '
            'runtime, open a fresh runtime, then re-run §1-§3, §5, §6, §8 '
            '(SKIP §4).'
        )

    # Wipe Colab's preinstalled huggingface_hub before llmcompressor's
    # pip install pulls its own version. Otherwise pip overlays the new
    # files on top of the old ones and you get a frankenstein hf_hub.
    print('Wiping preinstalled huggingface_hub for clean install...')
    get_ipython().run_line_magic('pip', 'uninstall -y -q huggingface_hub')
    import shutil, glob
    for d in glob.glob('/usr/local/lib/python3.12/dist-packages/huggingface_hub*'):
        shutil.rmtree(d, ignore_errors=True)

    print('Installing llmcompressor (sparsifier runtime)...')
    get_ipython().run_line_magic('pip', 'install -q --no-cache-dir "llmcompressor==0.10.0.2"')

    # Purge stale module cache so the next import reads freshly installed files.
    print('Purging stale module cache...')
    for k in [k for k in list(sys.modules)
              if k == 'huggingface_hub' or k.startswith('huggingface_hub.')
              or k == 'transformers' or k.startswith('transformers.')
              or k == 'datasets' or k.startswith('datasets.')]:
        del sys.modules[k]

    from llmcompressor import oneshot
    from llmcompressor.modifiers.pruning import WandaPruningModifier
    from transformers import AutoTokenizer

    print(f'\nRunning Wanda on {DENSE_HF_ID} → {WANDA_OUTPUT_DIR}')
    print('(~30-60 min on A100 for 512 calibration samples)\n')

    # save_compressed=False is critical. With the default (True), llmcompressor
    # writes sparse-bitmask shards but does NOT add a vLLM-compatible
    # quantization_config to config.json. vLLM 0.20 then sees `*.bitmask`
    # tensors with no compressor declared and KeyErrors on load. Saving
    # dense sidesteps the bitmask question entirely — the weights still
    # have the 2:4 zero pattern, vLLM just sees them as ordinary fp16 with
    # 50% zeros. Trade-off: vLLM's Sparse-Tensor-Core kernel won't auto-engage,
    # so tok/s will be ~equal to dense vLLM. Use Path A (RedHatAI checkpoint)
    # if you need the kernel-speedup measurement (§14 explains).
    model = oneshot(
        model                   = DENSE_HF_ID,
        dataset                 = 'open_platypus',
        recipe                  = WandaPruningModifier(
            sparsity        = 0.5,
            mask_structure  = '2:4',
            targets         = ['Linear'],
        ),
        output_dir              = str(WANDA_OUTPUT_DIR),
        max_seq_length          = 2048,
        num_calibration_samples = 512,
        save_compressed         = False,
    )

    # Belt-and-suspenders: if the running llmcompressor version ignores
    # save_compressed in oneshot, re-save explicitly. No-op if the kwarg
    # already took effect.
    if model is not None and hasattr(model, 'save_pretrained'):
        try:
            model.save_pretrained(str(WANDA_OUTPUT_DIR),
                                  safe_serialization=True,
                                  save_compressed=False)
        except TypeError:
            model.save_pretrained(str(WANDA_OUTPUT_DIR), safe_serialization=True)
        AutoTokenizer.from_pretrained(DENSE_HF_ID).save_pretrained(str(WANDA_OUTPUT_DIR))

    # Sanity-check the saved checkpoint: NO bitmask tensors, ~50% zeros
    # in MLP weights. If either fails, the bitmask format slipped through
    # and the eval runtime will crash with KeyError on load.
    import os
    from safetensors import safe_open
    shards = [f for f in os.listdir(WANDA_OUTPUT_DIR) if f.endswith('.safetensors')]
    n_bitmask, sample, zeros_pct = 0, None, None
    for shard in shards:
        with safe_open(os.path.join(WANDA_OUTPUT_DIR, shard), framework='pt') as s:
            keys = list(s.keys())
            n_bitmask += sum(1 for k in keys if 'bitmask' in k)
            if sample is None:
                cand = next((k for k in keys if k.endswith('mlp.down_proj.weight')), None)
                if cand:
                    sample = cand
                    zeros_pct = (s.get_tensor(cand) == 0).float().mean().item() * 100
    print(f'\n  bitmask tensors: {n_bitmask}  (must be 0)')
    if sample:
        print(f'  zeros in {sample}: {zeros_pct:.1f}%  (expect ~50% for 2:4)')
    assert n_bitmask == 0, ('Checkpoint still has bitmask tensors — save_compressed flag '
                             'did not take. Re-save dense manually with '
                             'model.save_pretrained(..., save_compressed=False) before '
                             'switching to the eval runtime.')
    if zeros_pct is not None:
        assert 45 < zeros_pct < 55, ('Sparsity is not ~50% — Wanda did not produce a '
                                      '2:4 mask. Investigate before evaluating.')

    print(f'\n✓ Wanda checkpoint saved (dense, 2:4 zero pattern) to {WANDA_OUTPUT_DIR}')
    print()
    print('NEXT STEP — switch to the eval runtime:')
    print('  1. Runtime → Disconnect and delete runtime.')
    print('  2. Reconnect to a fresh runtime.')
    print('  3. Run §1-§4 (Drive, GPU, paths, vllm install).')
    print('  4. Run §5 (HF login), §6 (still MODE="RUN_WANDA"), §7 (picks')
    print('     up the checkpoint you just saved), §10-§13 (helper, evals,')
    print('     comparison). SKIP §8 in the eval runtime.')


## §9 Shared latency prompts (same as `colab_standalone.ipynb` so cross-comparison is apples-to-apples)

In [ ]:
LATENCY_PROMPTS = [
    'Explain quantum entanglement to a high-school student in two sentences.',
    'Write a short Python function that returns the n-th Fibonacci number.',
    'Summarize the plot of Hamlet in three sentences.',
    'List five risks of deploying an LLM as a customer-support agent.',
    "Translate to French: 'The quick brown fox jumps over the lazy dog.'",
    "What is the capital of Australia, and why isn't it Sydney?",
    'Give one real-world example of a Markov chain.',
    'Write a haiku about debugging.',
]
LATENCY_MAX_TOKENS = 64

## §10 Helper: measure accuracy + latency for a vLLM-served model

Same shape as `run_method` in `colab_standalone.ipynb` — same return contract, same JSON schema, so the comparison cell at the end can stitch results from both notebooks together.

**Accuracy via lm-eval-harness subprocess.** vLLM holds GPU memory until process exit, so loading two `LLM()` instances in one Python process double-allocates. Subprocess isolation gives us a fresh GPU between accuracy and latency.

**Latency in-process** (after accuracy finishes and the subprocess GPU is freed).

In [ ]:
import subprocess, time, json, gc
import torch
from pathlib import Path

def _accuracy_via_lmeval(model_id, alias, task, limit, gpu_mem_util):
    """Run lm-eval-harness with vllm backend in a subprocess. Returns the
    parsed results JSON. Subprocess isolation keeps vLLM's GPU memory state
    separate from the in-process latency LLM that follows."""
    out_dir = RESULTS_DIR / 'lm_eval_vllm' / alias
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        'lm_eval', '--model', 'vllm',
        '--model_args',
        f'pretrained={model_id},dtype=float16,'
        f'gpu_memory_utilization={gpu_mem_util},trust_remote_code=True',
        '--tasks', task,
        '--num_fewshot', '0',
        '--batch_size', '1',
        '--output_path', str(out_dir),
    ]
    if limit is not None:
        cmd += ['--limit', str(limit)]
    print(f'$ {" ".join(cmd)}')
    subprocess.run(cmd, check=True)
    candidates = sorted(out_dir.glob('**/results*.json'),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise RuntimeError(f'lm-eval produced no results JSON under {out_dir}')
    with open(candidates[0]) as fh:
        return json.load(fh)

def _latency_in_process(model_id, gpu_mem_util):
    """Load the model with vllm.LLM and time greedy generation on the fixed
    prompt set. Frees the LLM at the end so the next call starts clean."""
    from vllm import LLM, SamplingParams
    print(f'Loading {model_id} into vLLM (gpu_memory_utilization={gpu_mem_util})')
    llm = LLM(
        model                  = model_id,
        dtype                  = 'float16',
        gpu_memory_utilization = gpu_mem_util,
        trust_remote_code      = True,
        download_dir           = str(HF_CACHE),
    )
    sampling = SamplingParams(temperature=0.0, max_tokens=LATENCY_MAX_TOKENS)
    llm.generate(LATENCY_PROMPTS[:1], sampling)   # warm-up
    t0 = time.perf_counter()
    outputs = llm.generate(LATENCY_PROMPTS, sampling)
    elapsed = time.perf_counter() - t0
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    tps = total_tokens / elapsed

    del llm
    gc.collect(); torch.cuda.empty_cache()
    return {
        'wall_clock_s'        : elapsed,
        'total_output_tokens' : int(total_tokens),
        'tokens_per_sec'      : tps,
        'num_prompts'         : len(LATENCY_PROMPTS),
        'max_new_tokens'      : LATENCY_MAX_TOKENS,
    }

def measure_vllm(model_id, alias, method, task=TASK, limit=LIMIT,
                 gpu_mem_util=0.85):
    """End-to-end: accuracy via subprocess, then latency in-process,
    persist JSON to Drive at results/<alias>.json."""
    print(f'\n=== {alias} ({method}) ===')
    print(f'Model: {model_id}')

    acc = _accuracy_via_lmeval(model_id, alias, task, limit, gpu_mem_util)
    print('\nAccuracy report:')
    print(json.dumps(acc.get('results', {}), indent=2))

    lat = _latency_in_process(model_id, gpu_mem_util)
    print(f'\nLatency: {lat["tokens_per_sec"]:.2f} tok/s '
          f'({lat["total_output_tokens"]} tokens in {lat["wall_clock_s"]:.2f} s)')

    result = {
        'alias'    : alias,
        'hf_id'    : model_id,
        'task'     : task,
        'limit'    : limit,
        'method'   : method,
        'accuracy' : acc,
        'latency'  : lat,
    }
    out = RESULTS_DIR / f'{alias}.json'
    out.write_text(json.dumps(result, indent=2, default=str))
    print(f'\nWrote {out}')
    return result

## §11 Dense baseline on vLLM

Llama-3.1-8B (or Mistral-7B-v0.3) BASE served by vLLM, no sparsity. This is the apples-to-apples baseline for the Wanda comparison: same engine, same prompts, same task, **same model family**, only sparsity differs.

Note we use the BASE model, not the Instruct variant, because the sparse checkpoints (RedHatAI / neuralmagic) are typically based on the base model. Using the Instruct variant on the dense side and a base sparse checkpoint conflates instruct-tuning with sparsity — that's how the original notebook ended up with the sparse model "outperforming" dense.

In [ ]:
dense_vllm = measure_vllm(
    model_id = DENSE_HF_ID,
    alias    = f'{MODEL_FAMILY}_dense_vllm',
    method   = 'vllm_dense',
)

## §12 Wanda 2:4 sparse on vLLM

Same evaluation pipeline, sparse model. vLLM auto-detects the 2:4 sparsity from the checkpoint's `compressed-tensors` metadata and dispatches to the Marlin kernel for sparse-aware GEMM. **No flag needed** — the engine reads the checkpoint, sees the sparsity declaration, and switches kernels automatically.

If you see `2:4 sparse path enabled` or similar in vLLM's startup log, you're getting the Sparse Tensor Core path. If not, vLLM is running dense over the masked weights — which means no speedup.

In [ ]:
sparse_vllm = measure_vllm(
    model_id = SPARSE_HF_ID,
    alias    = f'{MODEL_FAMILY}_wanda_2of4_vllm',
    method   = 'wanda_2of4_vllm',
)

## §13 Comparison: dense-vLLM vs Wanda-2of4-vLLM (and pull in R-Sparse from the other notebook)

Reads all four JSONs from `results/`, prints a unified table.

Caveat about cross-engine comparison: the **HuggingFace baseline** in `llama3_baseline.json` and the **vLLM baseline** here both run the same dense model — but vLLM uses different attention kernels (Flash-Attention, paged attention) and different scheduling. Apples-to-apples comparisons:
- Dense-HF vs R-Sparse-HF: valid (both HF engine)
- Dense-vLLM vs Wanda-vLLM: valid (both vLLM engine)
- Dense-HF vs Dense-vLLM: shows engine difference
- R-Sparse-HF vs Wanda-vLLM: NOT directly comparable (different engines AND different sparsity methods)

In [ ]:
import json
import pandas as pd
from pathlib import Path

def load(name):
    p = RESULTS_DIR / f'{name}.json'
    return json.loads(p.read_text()) if p.exists() else None

def extract_acc(report, task):
    if not report: return None
    raw = report.get('accuracy')
    if not raw: return None
    results = raw.get('results') if isinstance(raw, dict) else None
    if not results: return None
    metrics = results.get(task) or next(iter(results.values()), {})
    for key in ('acc_norm,none', 'acc_norm', 'acc,none', 'acc'):
        if key in metrics:
            return metrics[key]
    for v in metrics.values():
        if isinstance(v, (int, float)):
            return v
    return None

def extract_tps(report):
    if not report or 'latency' not in report: return None
    return report['latency'].get('tokens_per_sec')

# Pull every available result. Aliases match what colab_standalone.ipynb
# wrote (HF dense, R-Sparse) and what THIS notebook writes (vLLM dense,
# Wanda 2:4 vLLM). Missing files render as None — partial picture is fine.
ROW_TEMPLATE = [
    ('{family}_baseline',         'dense (HuggingFace)',   'HF'),
    ('{family}_rsparse',          'R-Sparse (HuggingFace)','HF'),
    ('{family}_dense_vllm',       'dense (vLLM)',          'vLLM'),
    ('{family}_wanda_2of4_vllm',  'Wanda 2:4 (vLLM)',      'vLLM'),
]

rows = []
for family in ['llama3', 'mistral', 'llama32_3b']:
    family_has_data = False
    for alias_template, label, engine in ROW_TEMPLATE:
        alias = alias_template.format(family=family)
        r = load(alias)
        if r is None:
            continue
        family_has_data = True
        rows.append({
            'family'       : family,
            'method'       : label,
            'engine'       : engine,
            f'{TASK} acc_norm' : extract_acc(r, TASK),
            'tok/s'        : extract_tps(r),
        })
    if not family_has_data:
        print(f'(no JSONs found on Drive for family={family})')

df = pd.DataFrame(rows)
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(df)

# Within-engine speedups, per family
print('\nWithin-engine speedup vs that engine\'s dense baseline:')
for family in df['family'].unique():
    sub = df[df['family'] == family].set_index('method')
    print(f'\n  {family}:')
    if 'dense (HuggingFace)' in sub.index and 'R-Sparse (HuggingFace)' in sub.index:
        d = sub.loc['dense (HuggingFace)', 'tok/s']
        r = sub.loc['R-Sparse (HuggingFace)', 'tok/s']
        if d and r:
            print(f'    R-Sparse on HF:    {r/d:.2f}x')
    if 'dense (vLLM)' in sub.index and 'Wanda 2:4 (vLLM)' in sub.index:
        d = sub.loc['dense (vLLM)', 'tok/s']
        w = sub.loc['Wanda 2:4 (vLLM)', 'tok/s']
        if d and w:
            print(f'    Wanda 2:4 on vLLM: {w/d:.2f}x')

# Save consolidated summary
summary_path = RESULTS_DIR / 'cross_method_summary.json'
summary_path.write_text(json.dumps({'task': TASK, 'limit': LIMIT, 'rows': rows},
                                    indent=2, default=str))
print(f'\nWrote {summary_path}')

## §14 What you should expect to see (sanity check)

On A100 40 GB, Llama-3.1-8B base, full PIQA (~1.8k examples).

**Path A — RedHatAI / Neural Magic pre-sparsified checkpoint:**

| method | engine | piqa `acc_norm` | tok/s | within-engine speedup |
|---|---|---:|---:|---:|
| dense (HF) | HF generate | ~0.81 | ~180-200 | (baseline) |
| R-Sparse (HF) | HF generate | ~0.79 | ~65 | 0.36× *(slower — kernel gap, see latency analysis doc)* |
| dense (vLLM) | vLLM | ~0.81 | ~600-1000 | (baseline) |
| Sparse-Llama 2:4 (vLLM) | vLLM | ~0.80 | ~900-1500 | **~1.4-1.7× faster** |

**Path B — your own Wanda run via §8 (saved dense, not bitmask):**

| method | engine | piqa `acc_norm` | tok/s | within-engine speedup |
|---|---|---:|---:|---:|
| dense (vLLM) | vLLM | ~0.81 | ~600-1000 | (baseline) |
| Wanda 2:4 (vLLM, dense save) | vLLM | ~0.78-0.80 | ~600-1000 | **~1.0× (no kernel dispatch — see below)** |

Three things to read off:
1. **vLLM dense is already 3-5× faster than HF dense** for the same model — that's paged-attention + Flash-Attention + better scheduling, nothing to do with sparsity.
2. **Path A delivers the kernel speedup** (~1.5×) because the RedHatAI checkpoint declares `sparse-bitmask` + `2:4` in its `quantization_config`, which vLLM 0.20 routes through compressed-tensors and dispatches to the Marlin24 kernel.
3. **Path B does NOT deliver the kernel speedup**, by design. §8 saves the Wanda-pruned weights as plain dense fp16 (with 50% zeros) so vLLM can load them at all. vLLM has no way to detect the 2:4 pattern from dense shards — it runs the dense kernel over zeros, getting no GEMM speedup. The accuracy number is still meaningful: it measures the Wanda algorithm. For the speedup measurement, use Path A.

**Why Path B saves dense:** the alternative is `save_compressed=True` → bitmask shards. llmcompressor 0.10 writes those shards but does not write a vLLM-compatible `quantization_config` to `config.json`, so vLLM crashes with `KeyError: 'layers.0.mlp.down_proj.bitmask'` on load. Patching the config with the right schema is brittle across `compressed-tensors` versions; saving dense is reliable.

If your Path A `tok/s` for the sparse model is *equal or slower* than dense vLLM, the sparse kernel didn\'t engage. Common causes:
- Pre-Ampere GPU (T4 etc.) — no Sparse Tensor Cores. Switch runtime.
- vLLM version too old — needs ≥ 0.6 for `compressed-tensors` 2:4. Re-run §4.
- Checkpoint isn\'t actually 2:4 sparse — `cat config.json | python -m json.tool | grep sparsity_structure`.

**About comparing across base vs instruct:** if you swap `DENSE_HF_ID` to `meta-llama/Llama-3.1-8B-Instruct` to also score the chat variant, add `--apply_chat_template --fewshot_as_multiturn` to the lm_eval invocation in §10\'s helper — without those flags Instruct models score ~5-10 pp lower than base on raw multiple-choice tasks like PIQA, which has nothing to do with sparsity.